# 2025-10-29-graph-parsing.ipynb
In this notebook we perform an initial exploration of the scene graphs: 

- How to parse scene graphs from the LLM-generated text files.
- Converting them into a networkx graph.
- Exploring a few graphs and see if they make sense.

## Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
from ast import literal_eval
import re

## Dataloading

In [3]:
with open("data/gpt_gen_formatted.json", "r") as f:
    raw = json.load(f)
# Take the first element of the first conversation to check out that scene graph.
first_prompt = raw[0]['conversations'][0]['content']
display(first_prompt[:100])

scene_graph_text = re.findall(r"Scene graph:(.*)", first_prompt)[0]
display(scene_graph_text[:100])

graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
display(graph_dict.keys())

"task: I need a shovel. Is there one in the scene?Scene graph:{'objects': [{'name': 'house_1', 'coord"

"{'objects': [{'name': 'house_1', 'coords': [-1, -1]}, {'name': 'house_2', 'coords': [-3, -1]}, {'nam"

dict_keys(['objects', 'regions', 'object_connections', 'region_connections', 'robot_location'])

## Using SPINE's utility to parse the graph.

In [4]:
# Turn the graph dict into a graph object
from spine.mapping.graph_util import parse_graph
parse_graph(graph_dict)

KeyError: 'shed_2'

Interesting... There's an edge referencing to a node that is not present in the graph.
Is this an underspecifid graph or a bug?

Let's parse all the graphs and find out.

## Parsing all the graphs

In [ ]:
prompt="""No scene graph found in prompt: task: How many robots are currently observed, where are they located, and which is the southmost robot?scene graph: {'objects': [{'name': 'office_building_1', 'coords': [200, 51]}, {'name': 'office_building_2', 'coords': [190, 50]}, {'name': 'robot_1', 'coords': [210, 65]}, {'name': 'robot_2', 'coords': [199, 65]}, {'name': 'robot_3', 'coords': [190, 70]}, {'name': 'example_truck_1', 'coords': [180, 70]}], 'regions': [{'name': 'example_sidewalk_1', 'coords': [180, 51]}, {'name': 'example_road_1', 'coords': [190, 50]}, {'name': 'charging_station_1', 'coords': [205, 60]}, {'name': 'charging_station_2', 'coords': [190, 68]}, {'name': 'example_node_2', 'coords': [150, 38]}, {'name': 'example_node_1', 'coords': [130, 38]}], 'object_connections': [['office_building_1', 'example_sidewalk_1'], ['office_building_2', 'example_sidewalk_1    '], ['robot_1', 'charging_station_1'], ['robot_2', 'charging_station_1'], ['robot_3', 'charging_station_2'], ['example_truck_1', 'charging_station_2']], 'region_connections': [['example_sidewalk_1', 'example_road_1'], ['example_sidewalk_1', 'charging_station_1'], ['example_sidewalk_1', 'charging_station_2']], 'robot_location': 'example_road_1'}
"""
re.search(pattern="[Ss]cene graph:", string=prompt) is not None

True

In [ ]:
# Take the first element of the first conversation to check out that scene graph.
import prism.scene_graph_parser as scene_graph_parser
scene_graph_dicts = []
pattern = "[Ss]cene graph:"
for conversation in raw:
    # Graph is in the first message
    prompt = conversation['conversations'][0]['content']
    if re.search(pattern=pattern, string=prompt):
        scene_graph_text = re.findall(pattern + r" ?(.*)", prompt)[0]
        graph_dict = literal_eval(scene_graph_text)  # handles the single quotes safely
        scene_graph_dicts.append(graph_dict)
    else: 
        raise ValueError(f"No scene graph found in prompt: {prompt}")
print(f"total scene graphs: {len(scene_graph_dicts)}")
display(scene_graph_dicts[0])

total scene graphs: 990


{'objects': [{'name': 'house_1', 'coords': [-1, -1]},
  {'name': 'house_2', 'coords': [-3, -1]},
  {'name': 'grocery_store_1', 'coords': [-5, -1]},
  {'name': 'shed_1', 'coords': [1, 3]},
  {'name': 'shed_1', 'coords': [1, 5]}],
 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]},
  {'name': 'example_road_2', 'coords': [-2, 0]},
  {'name': 'field_11', 'coords': [0, 1]},
  {'name': 'field_13', 'coords': [2, 3]}],
 'object_connections': [['house_1', 'example_road_1'],
  ['house_2', 'example_road_2'],
  ['shed_1', 'field_11'],
  ['shed_2', 'field_13']],
 'region_connections': [['example_road_1', 'example_road_2'],
  ['example_road_1', 'field_11'],
  ['field_11', 'field_13']],
 'robot_location': 'example_road_1'}

In [ ]:
# Take the first element of the first conversation to check out that scene graph.
import prism.scene_graph_parser as scene_graph_parser
scene_graph_dicts = []
pattern = "[Ss]cene graph:"
for conversation in raw:
    scene_graph_dicts.append(scene_graph_parser._parse_scene_graph_dictionary_from_conversation(conversation))
print(f"total scene graphs: {len(scene_graph_dicts)}")
display(scene_graph_dicts[0])

total scene graphs: 990


{'objects': [{'name': 'house_1', 'coords': [-1, -1]},
  {'name': 'house_2', 'coords': [-3, -1]},
  {'name': 'grocery_store_1', 'coords': [-5, -1]},
  {'name': 'shed_1', 'coords': [1, 3]},
  {'name': 'shed_1', 'coords': [1, 5]}],
 'regions': [{'name': 'example_road_1', 'coords': [-1, 0]},
  {'name': 'example_road_2', 'coords': [-2, 0]},
  {'name': 'field_11', 'coords': [0, 1]},
  {'name': 'field_13', 'coords': [2, 3]}],
 'object_connections': [['house_1', 'example_road_1'],
  ['house_2', 'example_road_2'],
  ['shed_1', 'field_11'],
  ['shed_2', 'field_13']],
 'region_connections': [['example_road_1', 'example_road_2'],
  ['example_road_1', 'field_11'],
  ['field_11', 'field_13']],
 'robot_location': 'example_road_1'}

## Exploration

### How many of the graphs have edges for nonexistent nodes?

### Clone of `parse_graph` to debug
TO DO: Modify this so that it handles the case where the graph is not correctly defined (missing nodes in edge list etc.)

In [ ]:
from typing import Dict, Optional, Tuple
from scipy.spatial.transform import Rotation
import networkx as nx
import numpy as np
from copy import deepcopy
from spine.mapping.graph_util import parse_graph_coord

def parse_graph(
    data: Dict[str, Dict[str, str]],
    custom_data: Optional[Dict[str, Dict[str, str]]] = {},
    rotation: Optional[Rotation] = None,
    utm_origin: Optional[np.ndarray] = None,
    flip_coords=False,
) -> Tuple[nx.Graph, str]:
    """Parse scene graph in `data` into a networkx object.

    Parameters
    ----------
    data : Dict[str, Dict[str, str]]
        graph where keys-values are nodes-attributes
    rotation : Optional[Rotation]
        current rotation of robot

    Returns
    -------
    Tuple[nx.Graph, str]
        Networkx and string of json
    """
    origin = np.array([0, 0])
    data = deepcopy(data)  # don't modify input data
    as_str = str(data)

    if utm_origin is not None:
        origin = utm_origin

    if len(custom_data):
        add_keys = ["regions", "region_connections", "objects", "object_connections"]
        for key in add_keys:
            if key in data and key in custom_data:
                data[key].extend(custom_data[key])

    G = nx.Graph()
    for node in data["objects"]:
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)
        if flip_coords:
            raise ValueError()
            # print("flipping coords")
            coords = [coords[0], -coords[1]]

        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for node in data["regions"]:
        assert "coords" in node, node
        c = node["coords"]
        # print(f"node: {node}, coords: {c}")
        coords = parse_graph_coord(node["coords"], origin=origin, rotation=rotation)

        if flip_coords:
            raise ValueError
            # print("flipping coords")
            coords = [coords[0], -coords[1]]
        node.pop("coords")
        name = node.pop("name")
        G.add_node(name, coords=coords, type="object", **node)

    for edge in data["object_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="object", weight=dist)

    for edge in data["region_connections"]:
        c1 = G.nodes[edge[0]]["coords"]
        c2 = G.nodes[edge[1]]["coords"]
        # print(f"edge: {edge}, c1, c2: {c1}, {c2}")
        dist = np.linalg.norm(np.array(c1) - np.array(c2))
        G.add_edge(edge[0], edge[1], type="region", weight=dist)

    return G, as_str

In [ ]:
parsed_graphs = []
for scene_graph_dict in scene_graph_dicts: 
    try: 
        parsed_graphs.append(parse_graph(scene_graph_dict))
    except Exception as e: 
        print(f"Error parsing scene graph: {e}")
display(parsed_graphs[0])
print(f"Parsed {len(parsed_graphs)} graphs out of {len(scene_graph_dicts)}")

Error parsing scene graph: 'shed_2'


(<networkx.classes.graph.Graph at 0x7835577439a0>,
 "{'objects': [{'name': 'office_building_1', 'coords': [200, 51]}, {'name': 'office_building_2', 'coords': [190, 50]}, {'name': 'robot_1', 'coords': [210, 65]}, {'name': 'robot_2', 'coords': [199, 65]}, {'name': 'robot_3', 'coords': [190, 70]}, {'name': 'example_truck_1', 'coords': [180, 70]}], 'regions': [{'name': 'example_sidewalk_1', 'coords': [180, 51]}, {'name': 'example_road_1', 'coords': [190, 50]}, {'name': 'charging_station_1', 'coords': [205, 60]}, {'name': 'charging_station_2', 'coords': [190, 68]}, {'name': 'example_node_2', 'coords': [150, 38]}, {'name': 'example_node_1', 'coords': [130, 38]}], 'object_connections': [['office_building_1', 'example_sidewalk_1'], ['office_building_2', 'example_sidewalk_1'], ['robot_1', 'charging_station_1'], ['robot_2', 'charging_station_1'], ['robot_3', 'charging_station_2'], ['example_truck_1', 'charging_station_2']], 'region_connections': [['example_sidewalk_1', 'example_road_1'], ['examp

Parsed 989 graphs out of 990


#### Sanity check

Sanity check: are there any other broken graphs that weren't dropped?

In [ ]:
import prism.scene_graph_parser as scene_graph_parser

count_undefined = 0
for scene_graph_dict in scene_graph_dicts: 
    undefined_nodes = scene_graph_parser.find_undefined_nodes(scene_graph_dict)
    if undefined_nodes: 
        count_undefined += 1
        print(f"Undefined nodes, dropping: {undefined_nodes}")

print(f"Graphs with undefined nodes: {count_undefined} out of {len(scene_graph_dicts)}")

Undefined nodes, dropping: ['shed_2']
Graphs with undefined nodes: 1 out of 990


## Adding to the dataset

In [ ]:
import datasets

dataset = datasets.load_dataset("json",data_files="data/gpt_gen_formatted.json")
def _add_scene_graph_feature(row):
    row["scene_graph"] = scene_graph_parser._parse_scene_graph_dictionary_from_conversation(row)
    return row

def _no_undefined_nodes(row):
    undefined_nodes = scene_graph_parser.find_undefined_nodes(row["scene_graph"])
    return len(undefined_nodes) == 0

dataset_w_graph = dataset.map(_add_scene_graph_feature).filter(_no_undefined_nodes)

## turning it into a PyG object
This only works outside of trl. In trl we'll have to take the scene_graph_dict and turn it into parsed_graph -> PyG inside architecture.

In [ ]:
import torch_geometric.utils as pyg_utils
import torch

pyg_graphs = []
for (nx_graph, scene_graph_dict) in parsed_graphs: 
    pyg_graph = pyg_utils.from_networkx(nx_graph)
    pyg_graph.x = torch.zeros(pyg_graph.num_nodes, 1)
    pyg_graphs.append(pyg_graph)

pyg_graphs[0]

Data(edge_index=[2, 18], coords=[12, 2], type=[12], edge_type=[18], weight=[18], num_nodes=12, x=[12, 1])

## Testing out the RPEARL Class

In [ ]:
from torch import nn
from torch.utils.checkpoint import checkpoint
from torch_geometric.nn import TAGConv
from torch_geometric.data import Data

class GCN(nn.Module):
    """
    A simple TAG-based graph convolutional backbone that returns node embeddings.

    Args:
        in_channels (int): Number of input features per node
        hidden_channels (int): Number of hidden features per node
        num_layers (int): Number of convolution layers (must be >= 2)
        skip_connection (bool): Whether to use skip connections
        k (int): Order of TAGConv polynomial (K)

    Returns:
        torch.Tensor: Node embeddings of shape [num_nodes, hidden_channels]
    """

    def __init__(
        self,
        in_channels,
        hidden_channels,
        num_layers,
        skip_connection=False,
        dropout=0.5,
        k: int = 3,
    ):
        super().__init__()
        if num_layers < 2:
            raise ValueError("GCN requires at least 2 layers.")

        self.convs = nn.ModuleList()
        self.k = k
        self.convs.append(TAGConv(in_channels, hidden_channels, K=self.k))
        self.norms = nn.ModuleList()
        for _ in range(num_layers - 2):
            self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
            self.norms.append(nn.LayerNorm(hidden_channels))
            # self.norms.append(nn.BatchNorm1d(hidden_channels))
        self.convs.append(TAGConv(hidden_channels, hidden_channels, K=self.k))
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=dropout)
        self.skip_connection = skip_connection
        self.embedding_dim = hidden_channels
        
    def forward(self, data: Data):
        """
        Forward pass through the GCN.

        Args:
            data (Data): PyTorch Geometric Data object containing node features (x)
                        and edge indices (edge_index)

        Returns:
            torch.Tensor: Output node embeddings [num_nodes, hidden_channels]
        """
        x0, edge_index = data.x, data.edge_index
        x_prev = x0
        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x_prev, edge_index)
            if i < len(self.norms):
                x = self.norms[i](x)
            x = self.relu(x)
            x = self.dropout(x)
            if self.skip_connection and i > 0:
                x = x + x_prev
            x_prev = x
        x = self.convs[-1](x, edge_index)
        return x

class DataGNNPositionalEncodings(nn.Module):
    """
    Graph positional encodings using the graph's true node features.

    Args:
        pe_hidden_channels (int): Hidden dimension for the GCN
        pe_num_layers (int): Number of layers in the GCN
        d_model (int): Output dimension
    """

    def __init__(
        self, 
        in_features, 
        pe_hidden_channels, 
        pe_num_layers, 
        d_model, 
        dropout=0.1
    ):
        super().__init__()
        # Create a GCN that takes d_model features and outputs d_model features
        # TODO support different GNNs for PEs.
        self.pe_gcn = GCN(
            in_features,
            pe_hidden_channels,
            pe_num_layers,
            skip_connection=True,
            dropout=dropout,
        )
        # Add a final projection to ensure output is d_model dimensions
        self.output_projection = nn.Linear(pe_hidden_channels, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, data):
        pe = self.pe_gcn(data)
        pe = self.dropout(pe)
        pe = self.output_projection(pe)
        return pe

class RandomGNNPositionalEncodings(nn.Module):
    """
    Random graph positional encodings (R-PEARL).

    Args:
        pe_hidden_channels (int): Hidden dimension for the GCN
        pe_num_layers (int): Number of layers in the GCN
        d_model (int): Output dimension
        num_samples (int): Number of random samples (M) to use
    """

    def __init__(
        self, pe_hidden_channels, pe_num_layers, d_model, num_samples=30, dropout=0.1
    ):
        super().__init__()
        # Create a GCN that takes 1-dimensional random features
        self.pe_gcn = GCN(
            1, pe_hidden_channels, pe_num_layers, skip_connection=True, dropout=dropout
        )
        # Add a final projection to ensure output is d_model dimensions
        self.output_projection = nn.Linear(pe_hidden_channels, d_model)
        self.dropout = nn.Dropout(dropout)
        # self.layer_norm = nn.LayerNorm(d_model)
        self.batch_norm = nn.BatchNorm1d(d_model)
        self.M = num_samples

    def forward(self, data):
        X, edge_index = data.x, data.edge_index

        # Generate random node embeddings for positional encoding
        num_nodes = X.shape[0]
        Q = torch.randn((num_nodes, self.M), device=X.device)

        # Process random embeddings individually through GCN
        P_m = []

        for i in range(self.M):

            def _pe_block(q_col, edge_idx, _dummy):
                q_data = Data(x=q_col.unsqueeze(-1), edge_index=edge_idx)
                pe_local = self.pe_gcn(q_data)
                pe_local = self.dropout(pe_local)
                pe_local = self.output_projection(pe_local)
                return pe_local

            dummy = Q.new_ones(1, requires_grad=True)
            pe = checkpoint(_pe_block, Q[:, i], edge_index, dummy)
            P_m.append(pe)
        # checkpoint

        P = torch.stack(P_m, dim=-1)
        pooled_pe = P.mean(dim=-1)
        # pooled_pe = self.layer_norm(pooled_pe)
        pooled_pe = self.batch_norm(pooled_pe)
        return pooled_pe


pe_model = f(
    pe_hidden_channels=256, pe_num_layers=3, d_model=256, num_samples=10, dropout=0.1
)
pe_model(pyg_graphs[0])

/home/jporras/miniconda3/envs/prism/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


tensor([[-0.1825, -0.5903,  0.9684,  ...,  0.4321, -0.2345, -0.4024],
        [ 0.0847, -1.0437,  1.4094,  ...,  0.0104, -0.0105,  0.2196],
        [-0.2461, -0.1713, -0.4870,  ...,  0.7957, -0.0125,  0.4463],
        ...,
        [ 1.1577,  1.0986,  1.9875,  ...,  1.7913,  0.9098,  2.5262],
        [-1.3757, -1.6416, -1.1691,  ..., -1.3619, -1.6936,  0.7875],
        [-1.6058, -1.0401, -0.8858,  ..., -1.3183, -1.7488, -0.1758]],
       grad_fn=<NativeBatchNormBackward0>)

## LLM Wrapper with GNN Encoder

In [ ]:
from huggingface_hub import login,whoami

whoami()

{'type': 'user',
 'id': '635ac6aaf9a004065f3141f1',
 'name': 'jotaporras',
 'fullname': 'Porras-Valenzuela',
 'email': 'jporras06@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'periodEnd': None,
 'isPro': False,
 'avatarUrl': '/avatars/eb22f3ba332599c64b93a26eb211d2fa.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'lb1v2',
   'role': 'read',
   'createdAt': '2025-10-30T15:55:02.516Z'}}}

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
#model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")

model,tokenizer

Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


(Qwen2ForCausalLM(
   (model): Qwen2Model(
     (embed_tokens): Embedding(151936, 896)
     (layers): ModuleList(
       (0-23): 24 x Qwen2DecoderLayer(
         (self_attn): Qwen2Attention(
           (q_proj): Linear(in_features=896, out_features=896, bias=True)
           (k_proj): Linear(in_features=896, out_features=128, bias=True)
           (v_proj): Linear(in_features=896, out_features=128, bias=True)
           (o_proj): Linear(in_features=896, out_features=896, bias=False)
         )
         (mlp): Qwen2MLP(
           (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
           (up_proj): Linear(in_features=896, out_features=4864, bias=False)
           (down_proj): Linear(in_features=4864, out_features=896, bias=False)
           (act_fn): SiLU()
         )
         (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
         (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
       )
     )
     (norm): Qwen2RMSNorm((896,), eps=1e-06)
     (rotar

From `modular_qwen2.py`'s forward pass: 
```python
@deprecate_kwarg("past_key_value", new_name="past_key_values", version="4.58")
    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
        past_key_values: Optional[Cache] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs: Unpack[FlashAttentionKwargs],
    ) -> tuple[torch.Tensor, Optional[torch.Tensor]]:
```

In [ ]:
from trl.trainer import SFTTrainer, SFTConfig
from datasets import load_dataset
from prism import scene_graph_parser

train_dataset = load_dataset("json", data_files=["data/gpt_gen_formatted.json"], split="train")


def _add_messages(example):
    example["messages"] = example["conversations"]
    return example


def _tokenize_with_conversations(example):
    tokenized = tokenizer.apply_chat_template(
        example["messages"], tokenize=True, return_dict=True
    )
    tokenized["conversations"] = example["conversations"]
    tokenized["messages"] = example["messages"]
    return tokenized


train_dataset = train_dataset.map(_add_messages)
train_dataset = train_dataset.map(_tokenize_with_conversations)
# train_dataset = train_dataset.map(scene_graph_parser._parse_scene_graph_dictionary_from_conversation)

Map:   0%|          | 0/990 [00:00<?, ? examples/s]

In [ ]:
train_dataset


Dataset({
    features: ['conversations', 'messages', 'input_ids', 'attention_mask'],
    num_rows: 990
})

In [ ]:
from transformers.data.data_collator import DataCollatorForLanguageModeling
from prism.scene_graph_parser import (
    _parse_scene_graph_dictionary_from_conversation,
    find_undefined_nodes,
)
from spine.mapping.graph_util import parse_graph
import torch
import torch_geometric.utils as pyg_utils


class DataCollatorForGraphAugmentedLLM(DataCollatorForLanguageModeling):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def __call__(self, examples):
        """Attach parsed PyG graphs for each conversation example."""
        pyg_graphs = []
        sanitized_examples = []

        for example in examples:
            scene_graph_dict = _parse_scene_graph_dictionary_from_conversation(example)
            if find_undefined_nodes(scene_graph_dict):
                continue # Skip graphs where edge lists contain undefined nodes.

            nx_graph, _ = parse_graph(scene_graph_dict)
            node_names = list(nx_graph.nodes)
            coords = torch.tensor(
                [nx_graph.nodes[node]["coords"] for node in node_names],
                dtype=torch.float32,
            )

            pyg_graph = pyg_utils.from_networkx(nx_graph)
            pyg_graph.coords = coords
            pyg_graph.x = torch.zeros((coords.size(0), 1), dtype=torch.float32)
            pyg_graph.node_names = node_names
            pyg_graph.node_types = [nx_graph.nodes[node]["type"] for node in node_names]
            pyg_graph.robot_location = scene_graph_dict.get("robot_location")
            pyg_graph.raw_scene_graph = scene_graph_dict
            pyg_graphs.append(pyg_graph)

            sanitized_examples.append(
                {
                    k: v
                    for k, v in example.items()
                    if k not in {"conversations", "scene_graph", "messages"}
                }
            )
        # Call the parent collator to get the tensors (on sanitized examples so that it doesn't try to tensorize the scene graph/text)
        batch = super().__call__(sanitized_examples)
        batch["graphs"] = pyg_graphs
        return batch


collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)
collator([train_dataset[i] for i in range(4)])

{'input_ids': tensor([[151644,   8948,    198,  ...,     92, 151645,    198],
        [151644,   8948,    198,  ..., 151643, 151643, 151643],
        [151644,   8948,    198,  ..., 151643, 151643, 151643]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'labels': tensor([[151644,   8948,    198,  ...,     92, 151645,    198],
        [151644,   8948,    198,  ...,   -100,   -100,   -100],
        [151644,   8948,    198,  ...,   -100,   -100,   -100]]), 'graphs': [Data(
  edge_index=[2, 18],
  coords=[12, 2],
  type=[12],
  edge_type=[18],
  weight=[18],
  num_nodes=12,
  x=[12, 1],
  node_names=[12],
  node_types=[12],
  robot_location='example_road_1',
  raw_scene_graph={
    objects=[6],
    regions=[6],
    object_connections=[6],
    region_connections=[3],
    robot_location='example_road_1',
  }
), Data(
  edge_index=[2, 8],
  coords=[7, 2],
  type=[7],
  edge_type=[8],
  weight=[8],
  num_nodes=7,
  x=[

In [ ]:
# Quick SFT sanity check without graph augmentation
baseline_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


sft_baseline_config = SFTConfig(
    output_dir="/home/jporras/sourcecode/GREP-PRISM/output",
    max_steps=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    logging_steps=1,
    logging_strategy="steps",
    save_strategy="no",
    #evaluation_strategy="no",
    save_safetensors=False,
    report_to="none",
)

train_subset = train_dataset.select(range(8))

baseline_trainer = SFTTrainer(
    model=q,
    args=sft_baseline_config,
    train_dataset=train_subset,
    processing_class=tokenizer,
    data_collator=baseline_collator,
)

baseline_train_result = baseline_trainer.train()
baseline_train_result

Step,Training Loss
1,0.118300
2,0.124400
3,0.181000
4,0.145000
5,0.077900


TrainOutput(global_step=5, training_loss=0.12931467741727828, metrics={'train_runtime': 3.1077, 'train_samples_per_second': 3.218, 'train_steps_per_second': 1.609, 'total_flos': 19867639993344.0, 'train_loss': 0.12931467741727828, 'epoch': 1.25})

In [ ]:
class GraphAugmentedLLM(nn.Module):
    def __init__(self, llm: nn.Module, pe_model: nn.Module):
        super().__init__()
        self.llm = llm
        self.pe_model = pe_model
        self.config = llm.config

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)  # defer to nn.Module first
        except AttributeError:
            return getattr(self.llm, name)

    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        graphs: list | None = None,
        **kwargs,
    ):
        _ = graphs  # placeholder until we integrate the GNN outputs into the LLM
        print(graphs)
        return self.llm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            **kwargs,
        )
pe_model = RandomGNNPositionalEncodings(
    pe_hidden_channels=256, pe_num_layers=3, d_model=256, num_samples=10, dropout=0.1
)
graph_augmented_model = GraphAugmentedLLM(model, pe_model)

collator = DataCollatorForGraphAugmentedLLM(tokenizer=tokenizer, mlm=False)

baseline_trainer = SFTTrainer(
    model=graph_augmented_model,
    args=sft_baseline_config,
    train_dataset=train_subset,
    processing_class=tokenizer,
    data_collator=collator,
)

baseline_train_result = baseline_trainer.train()
baseline_train_result

ValueError: Conversation payload is missing the 'conversations' key.

None


Step,Training Loss
1,0.075700
2,0.043500
3,0.065500
4,0.070000
5,0.038200


None
None
None
None


TrainOutput(global_step=5, training_loss=0.058609000593423846, metrics={'train_runtime': 2.9788, 'train_samples_per_second': 3.357, 'train_steps_per_second': 1.679, 'total_flos': 19867639993344.0, 'train_loss': 0.058609000593423846, 'epoch': 1.25})